# TEC206 Intermediate Programming — Week 08
## Exceptions, Error Handling, Validation & Custom Exceptions

**Lecture format:** explanation → demonstration → flowchart → guided practice → challenge → self-check quiz  
**Language:** Python 3  
**Prerequisite:** variables, data types, conditionals, loops, functions, classes, and basic file handling

---

### Why this workbook exists

Programs do not always receive perfect input. Files can be missing, users can type text where a number is expected, list indexes can be invalid, and calculations can fail. A strong Python program should not simply crash when a predictable problem occurs.

This week we learn how Python represents these problems as **exceptions**, and how we can respond using:

- `try`
- `except`
- `else`
- `finally`
- `raise`
- built-in exception types
- custom exception classes

By the end of the workbook, you should be able to **recognise an error, identify the exception type, decide whether it should be handled, and write code that fails safely and clearly**.

> ## How to use this notebook
>
> 1. Run cells from top to bottom.
> 2. **Predict the result before running each demonstration.**
> 3. When an exception is shown, read the **exception type** and the **message** separately.
> 4. Change the input values and observe which branch executes.
> 5. Complete each activity before looking at the suggested solution.
> 6. The final quiz uses `ipywidgets` when available, but the rest of the notebook works without it.
>
> ### Important principle
>
> **Exceptions are not a replacement for good logic.** Use normal `if` statements for expected decisions and exception handling for operations that can genuinely fail.

# 1. Learning Outcomes

By the end of Week 08, you should be able to:

- distinguish **syntax errors** from **runtime errors**;
- explain what an **exception** is;
- recognise common built-in exceptions such as `NameError`, `TypeError`, `ValueError`, and `ZeroDivisionError`;
- use `try` and `except` to prevent predictable failures from crashing a program;
- catch **specific exceptions** rather than hiding every problem;
- use multiple `except` blocks when different failures require different responses;
- explain when the `else` block executes;
- explain why the `finally` block executes regardless of success or failure;
- inspect an exception with `except SomeError as exc`;
- deliberately trigger an exception using `raise`;
- create a **custom exception** for a domain-specific rule;
- validate age, dates, numeric input, and delivery distance;
- design a complete `try → except / else → finally` control flow.

# 2. Two Broad Classes of Programming Errors

For this week, we will use a useful introductory classification:

| Error class | When it is discovered | What it means | Example |
|---|---|---|---|
| **Syntax error** | Before the invalid statement can execute | The code does not follow Python grammar | Missing `:` after `if` |
| **Runtime error / exception** | While the program is running | Python reached valid syntax but could not complete an operation | `10 / 0` |

> In professional debugging you may also hear about **logic errors**: the program runs, but produces the wrong result. Logic errors are important, but this week our main focus is syntax errors and runtime exceptions.

## 2.1 Error Classification Flowchart

```text
                         PROGRAM PROBLEM
                               │
               ┌───────────────┴───────────────┐
               │                               │
        SYNTAX ERROR                    RUNTIME ERROR
    Python grammar is invalid          Code begins executing
               │                               │
     Program cannot execute              An operation fails
     the invalid statement                     │
               │                         Python raises an
       Fix the source code                  EXCEPTION
                                               │
                                  Handle it, prevent it, or
                                  allow it to stop the program
```

In [ ]:
# Visual version of the classification flowchart.
# No additional package beyond matplotlib is required.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")

nodes = {
    "Program problem": (0.50, 0.88),
    "Syntax error\nInvalid Python grammar": (0.25, 0.55),
    "Runtime error\nException while executing": (0.75, 0.55),
    "Fix the source code": (0.25, 0.20),
    "Handle / prevent / propagate\nthe exception": (0.75, 0.20),
}

for label, (x, y) in nodes.items():
    ax.text(x, y, label, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5"))

for start, end in [
    ((0.50, 0.82), (0.25, 0.63)),
    ((0.50, 0.82), (0.75, 0.63)),
    ((0.25, 0.46), (0.25, 0.28)),
    ((0.75, 0.46), (0.75, 0.28)),
]:
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->"))

plt.show()

# 3. Syntax Errors

A **syntax error** means Python cannot understand the structure of the code according to its grammar rules.

Common causes include:

- missing `:` after `if`, `for`, `while`, `def`, or `class`;
- unmatched brackets or parentheses;
- unterminated strings;
- incorrect indentation in a required block;
- writing tokens in an invalid order.

A syntax error is different from a normal runtime exception because the invalid code cannot be executed normally.

In [ ]:
# We use compile() so the notebook itself stays runnable while still
# demonstrating a SyntaxError safely.

bad_source = """
if 10 > 5
    print('Hello')
"""

try:
    compile(bad_source, "<student-example>", "exec")
except SyntaxError as exc:
    print("Exception type:", type(exc).__name__)
    print("Message       :", exc)

### Read the error message

When Python reports an error, look for two things:

1. **Exception type** — for example `SyntaxError`.
2. **Message** — Python's description of what went wrong.

The type tells you the broad category. The message gives context about this particular failure.

# 4. Runtime Errors and Exceptions

A **runtime error** happens after Python has begun executing valid code.

When an operation cannot be completed, Python creates and **raises an exception object**.

A simplified process is:

```text
Python executes a statement
        │
        ▼
Can the operation complete?
    ┌───┴───┐
   Yes      No
    │        │
 Continue   Python creates an exception
             │
             ▼
      Is matching handling available?
          ┌───┴───┐
         Yes      No
          │        │
      Run handler  Program stops and
                   traceback is displayed
```

In [ ]:
# Runtime exception example: division by zero

try:
    result = 10 / 0
except ZeroDivisionError as exc:
    print("Caught:", type(exc).__name__)
    print("Message:", exc)

# 5. Common Built-in Exceptions

Python contains many built-in exception classes. You do **not** need to memorise every exception today. Instead, learn to recognise the common ones and understand what kind of operation triggers them.

The table below focuses on exceptions you are likely to encounter in intermediate programming.

In [ ]:
# Build and display a table of common Python exceptions.

exception_rows = [
    {"Exception": "SyntaxError", "Category": "Parsing", "Typical cause": "Invalid Python syntax", "Purpose / signal": "Python cannot parse the source code"},
    {"Exception": "NameError", "Category": "Names", "Typical cause": "Using a name that has not been defined", "Purpose / signal": "Requested variable/function name is unknown"},
    {"Exception": "TypeError", "Category": "Types", "Typical cause": "Operation used with an inappropriate type", "Purpose / signal": "Object type is incompatible with the operation"},
    {"Exception": "ValueError", "Category": "Values", "Typical cause": "Correct type, unacceptable value", "Purpose / signal": "Value cannot be interpreted or accepted"},
    {"Exception": "ZeroDivisionError", "Category": "Arithmetic", "Typical cause": "Division or modulo by zero", "Purpose / signal": "Arithmetic operation is mathematically undefined"},
    {"Exception": "IndexError", "Category": "Sequences", "Typical cause": "List/string index is outside valid range", "Purpose / signal": "Requested sequence position does not exist"},
    {"Exception": "KeyError", "Category": "Mappings", "Typical cause": "Dictionary key is missing", "Purpose / signal": "Requested mapping key does not exist"},
    {"Exception": "AttributeError", "Category": "Objects", "Typical cause": "Object has no requested attribute/method", "Purpose / signal": "Attribute lookup failed"},
    {"Exception": "FileNotFoundError", "Category": "Files / OS", "Typical cause": "Opening a file that does not exist", "Purpose / signal": "Requested filesystem path was not found"},
    {"Exception": "PermissionError", "Category": "Files / OS", "Typical cause": "Operation is not permitted", "Purpose / signal": "Operating system denied access"},
    {"Exception": "ImportError", "Category": "Imports", "Typical cause": "Import cannot be completed", "Purpose / signal": "Requested import failed"},
    {"Exception": "ModuleNotFoundError", "Category": "Imports", "Typical cause": "Module cannot be located", "Purpose / signal": "Python cannot find the requested module"},
    {"Exception": "OverflowError", "Category": "Arithmetic", "Typical cause": "Numeric result exceeds supported range for an operation", "Purpose / signal": "Arithmetic result is too large for that operation"},
    {"Exception": "RuntimeError", "Category": "Runtime", "Typical cause": "Generic runtime condition without a more specific built-in type", "Purpose / signal": "A runtime problem occurred"},
    {"Exception": "Exception", "Category": "Base class", "Typical cause": "Used to catch many normal application exceptions", "Purpose / signal": "Parent class for most exceptions we normally handle"},
]

try:
    import pandas as pd
    from IPython.display import display
    exception_table = pd.DataFrame(exception_rows)
    display(exception_table)
except ImportError:
    # Standard-library fallback if pandas is unavailable.
    headers = list(exception_rows[0])
    widths = {h: max(len(h), max(len(str(row[h])) for row in exception_rows)) for h in headers}
    print(" | ".join(h.ljust(widths[h]) for h in headers))
    print("-+-".join("-" * widths[h] for h in headers))
    for row in exception_rows:
        print(" | ".join(str(row[h]).ljust(widths[h]) for h in headers))

## 5.1 A Simplified Exception Hierarchy

Python exceptions form a class hierarchy. This matters because catching a parent class also catches many child exception types.

```text
BaseException
│
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
│
└── Exception        ← normal application exceptions usually live here
    │
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    │
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    │
    ├── OSError
    │   ├── FileNotFoundError
    │   └── PermissionError
    │
    ├── TypeError
    ├── ValueError
    ├── NameError
    ├── AttributeError
    └── RuntimeError
```

> **Key idea:** `Exception` is broad. `ValueError`, `TypeError`, and `ZeroDivisionError` are more specific. Prefer the **most specific exception that you can meaningfully handle**.

# 6. Demonstration: Triggering Common Exceptions Safely

Normally an uncaught exception stops the current program flow. In the next demonstration, each example is wrapped so that we can inspect several exception types without stopping the notebook.

In [ ]:
def show_exception(label, action):
    """Run an action and print the exception type and message if it fails."""
    try:
        action()
    except Exception as exc:
        print(f"{label:<22} -> {type(exc).__name__}: {exc}")


show_exception("NameError", lambda: eval("not_defined_name"))
show_exception("TypeError", lambda: "5" + 2)
show_exception("ValueError", lambda: int("hello"))
show_exception("ZeroDivisionError", lambda: 10 / 0)
show_exception("IndexError", lambda: [10, 20][5])
show_exception("KeyError", lambda: {"name": "Aisha"}["age"])
show_exception("AttributeError", lambda: (10).append(2))
show_exception("FileNotFoundError", lambda: open("this_file_should_not_exist_tec206.txt"))

# 7. The Basic `try` / `except` Pattern

The simplest exception-handling structure is:

```python
try:
    # code that may fail
except SomeException:
    # recovery / error message
```

Python first executes the `try` block. If a matching exception occurs, Python **stops the remaining statements inside that `try` block** and jumps to the matching `except` block.

## 7.1 `try` / `except` Flowchart

```text
          ┌─────────────────────┐
          │ Enter try block     │
          └─────────┬───────────┘
                    │
                    ▼
          Execute statements in try
                    │
             ┌──────┴──────┐
             │ Exception?  │
             └──────┬──────┘
                No  │  Yes
              ┌─────┘   └─────────────┐
              ▼                       ▼
     Finish try normally       Does an except match?
              │                  ┌────┴────┐
              │                 Yes        No
              │                  │          │
              │                  ▼          ▼
              │              Run except   Propagate
              │                  │       exception
              └──────────────┬───┘
                             ▼
                     Continue after structure
```

In [ ]:
# Basic example: converting input to an integer

user_text = "27"

try:
    age = int(user_text)
    print("Conversion succeeded. Age =", age)
except ValueError:
    print("Please enter a whole number.")

## 7.2 Why `int()` can raise `ValueError`

`int()` accepts values that can be interpreted as integers:

```python
int("25")     # valid
int("-3")     # valid conversion
```

But this cannot be interpreted as a base-10 integer:

```python
int("twenty")
```

The input is still a string, so the problem is not that `int()` received a forbidden *type*. It received a string with an unacceptable *value* for integer conversion, so Python raises `ValueError`.

In [ ]:
for sample in ["25", "-3", "twenty", "19.5", ""]:
    try:
        number = int(sample)
        print(f"{sample!r:>10} -> success: {number}")
    except ValueError as exc:
        print(f"{sample!r:>10} -> ValueError: {exc}")

# 8. Catching the Exception Object with `as`

Often we want access to Python's original error message.

```python
except ValueError as exc:
    print(exc)
```

`exc` is simply a variable that refers to the exception object raised by Python.

In [ ]:
try:
    age = int("twenty-five")
except ValueError as exc:
    print("Type   :", type(exc).__name__)
    print("Message:", exc)

# 9. Specific Exceptions vs `except Exception`

Compare these two approaches:

```python
except Exception:
    print("Something went wrong")
```

and:

```python
except ValueError:
    print("The input must be an integer")
```

The second version is usually better because it handles the **problem you expected** without hiding unrelated bugs.

### Recommended rule

- Catch a **specific exception** when you know how to recover from it.
- Use `except Exception as exc` at a boundary where you genuinely need a broad catch, logging, or graceful shutdown.
- Avoid an empty bare `except:` in ordinary application code because it also catches exceptions such as `KeyboardInterrupt` and `SystemExit`.

In [ ]:
def parse_quantity(text):
    try:
        return int(text)
    except ValueError:
        print("Quantity must be a whole number.")
        return None

print(parse_quantity("12"))
print(parse_quantity("twelve"))

# 10. Multiple `except` Blocks

A single operation can fail for different reasons. Different exception types can be handled differently.

```python
try:
    ...
except ValueError:
    ...
except ZeroDivisionError:
    ...
```

Only the first matching handler executes.

In [ ]:
def safe_divide(numerator_text, denominator_text):
    try:
        numerator = float(numerator_text)
        denominator = float(denominator_text)
        answer = numerator / denominator
        print("Answer:", answer)
    except ValueError:
        print("Input error: both values must be numeric.")
    except ZeroDivisionError:
        print("Math error: the denominator cannot be zero.")


safe_divide("10", "2")
safe_divide("ten", "2")
safe_divide("10", "0")

# 11. The `else` Block

`else` belongs to the exception-handling structure and runs **only if the `try` block finishes without raising an exception**.

```python
try:
    risky_operation()
except SomeError:
    handle_error()
else:
    # Runs only after successful try
    continue_success_path()
```

Why use `else` instead of putting everything in `try`?

Because the `try` block should normally contain only the statements that are expected to raise the exceptions you are handling. This keeps error handling precise.

In [ ]:
def calculate_average(total_text, count_text):
    try:
        total = float(total_text)
        count = int(count_text)
        average = total / count
    except ValueError:
        print("Could not convert the input to numbers.")
    except ZeroDivisionError:
        print("Count cannot be zero.")
    else:
        print("Calculation succeeded.")
        print("Average =", average)


calculate_average("95", "5")
calculate_average("95", "0")
calculate_average("ninety-five", "5")

# 12. The `finally` Block

`finally` runs **whether an exception occurs or not**.

Typical uses include cleanup actions such as:

- closing a resource;
- releasing a lock;
- resetting temporary state;
- printing/logging that an operation has finished.

```python
try:
    ...
except SomeError:
    ...
finally:
    # runs in both success and failure paths
    ...
```

In [ ]:
def demonstrate_finally(text):
    print(f"\nTesting {text!r}")
    try:
        value = int(text)
        print("Inside try: conversion succeeded")
    except ValueError:
        print("Inside except: conversion failed")
    finally:
        print("Inside finally: this line always runs")


demonstrate_finally("42")
demonstrate_finally("forty-two")

## 12.1 Resource Cleanup Example

The following example uses `finally` to make sure a file handle is closed if it was successfully opened.

> In modern Python, `with open(...) as file:` is usually the cleaner way to manage files because the context manager handles cleanup automatically. We use `finally` here to understand the underlying control-flow idea.

In [ ]:
from pathlib import Path

path = Path("tec206_week08_demo.txt")
path.write_text("Exceptions make programs more robust.\n", encoding="utf-8")

file_handle = None

try:
    file_handle = open(path, "r", encoding="utf-8")
    print(file_handle.read())
except OSError as exc:
    print("File operation failed:", exc)
finally:
    if file_handle is not None:
        file_handle.close()
        print("File closed in finally.")

# Remove the temporary demo file.
path.unlink(missing_ok=True)

# 13. Complete `try` → `except` → `else` → `finally` Flow

This is the complete structure:

```python
try:
    # operation that might fail
except SpecificError:
    # runs if that error occurs
else:
    # runs only if try succeeds
finally:
    # runs in both cases
```

### Flowchart

```text
                     START
                       │
                       ▼
                  ┌─────────┐
                  │   try   │
                  └────┬────┘
                       │
               Was an exception raised?
                  ┌────┴────┐
                 No         Yes
                  │           │
                  ▼           ▼
               `else`     Matching `except`
                  │           │
                  └─────┬─────┘
                        ▼
                    `finally`
                        │
                        ▼
                 Continue / finish
```

### Remember

- **No exception:** `try` → `else` → `finally`
- **Matching exception:** `try` stops early → `except` → `finally`
- **Unmatched exception:** `finally` still runs, then the exception continues upward

In [ ]:
def full_flow(value_text):
    print(f"\nInput: {value_text!r}")
    try:
        print("1. TRY begins")
        value = int(value_text)
        print("2. TRY conversion completed")
    except ValueError as exc:
        print("3. EXCEPT runs ->", exc)
    else:
        print("3. ELSE runs -> value =", value)
    finally:
        print("4. FINALLY always runs")


full_flow("50")
full_flow("fifty")

# 14. Activity 1 — Age Input with `try` / `except`

### Task

Write a Python program that:

- prompts the user to enter their age;
- uses a `try/except` block to handle input errors;
- inside the `try` block, converts the input to an integer using `int()`;
- if conversion succeeds, prints a message containing the user's age;
- if conversion raises `ValueError`, catches it and prints an error message;
- is tested with both valid and invalid inputs.

### Predict before running

What should happen for:

- `25`
- `twenty five`
- `19.5`
- `-4`
- an empty input

In [ ]:
# Activity 1 — basic required version

try:
    age = int(input("Enter your age: "))
    print(f"Your age is {age}.")
except ValueError:
    print("Invalid input. Please enter your age as a whole number.")

## 14.1 Test the Same Logic Without Re-typing Input

Interactive `input()` is useful for users, but functions make automated testing much easier.

In [ ]:
def process_age(raw_age):
    try:
        age = int(raw_age)
        return f"Your age is {age}."
    except ValueError:
        return "Invalid input. Please enter your age as a whole number."


test_inputs = ["25", "twenty five", "19.5", "-4", ""]

for raw in test_inputs:
    print(f"Input {raw!r:>14} -> {process_age(raw)}")

## 14.2 Extension — Conversion Is Not the Same as Validation

`int("-4")` converts successfully, but a negative human age is usually invalid for our application.

Exception handling answers:

> **Could the operation be completed?**

Validation answers:

> **Is the resulting value acceptable for our program?**

We often need both.

In [ ]:
def validate_age(raw_age):
    try:
        age = int(raw_age)
    except ValueError:
        return "Invalid input: age must be a whole number."
    else:
        if not 0 <= age <= 130:
            return "Invalid age: please enter a value from 0 to 130."
        return f"Accepted age: {age}"


for raw in ["25", "twenty", "-4", "131", "19.5", "0"]:
    print(f"{raw!r:>10} -> {validate_age(raw)}")

# 15. Activity 2 — Division by Zero

Write a function that accepts two values, converts both to numbers, divides the first by the second, and handles:

- invalid numeric input with `ValueError`;
- division by zero with `ZeroDivisionError`;
- successful calculation with `else`;
- a completion message with `finally`.

In [ ]:
def division_activity(a_text, b_text):
    try:
        a = float(a_text)
        b = float(b_text)
        result = a / b
    except ValueError:
        print("Both inputs must be numbers.")
    except ZeroDivisionError:
        print("Cannot divide by zero.")
    else:
        print("Result =", result)
    finally:
        print("Division attempt complete.")


division_activity("20", "4")
division_activity("20", "0")
division_activity("twenty", "4")

# 16. Validating Dates with Integers and Exceptions

A date gives us a useful two-stage validation problem:

1. Can the user input be converted to integers?
2. Do those integers represent a real calendar date?

For example:

- `10 / 09 / 2026` is a real date;
- `31 / 02 / 2026` contains valid integers but is **not** a real calendar date.

The `datetime.date` constructor raises `ValueError` when the numeric values do not form a valid date.

In [ ]:
from datetime import date


def build_date(day_text, month_text, year_text):
    try:
        day = int(day_text)
        month = int(month_text)
        year = int(year_text)
        chosen_date = date(year, month, day)
    except ValueError as exc:
        print("Invalid date input:", exc)
    else:
        print("Valid date:", chosen_date.strftime("%A, %d %B %Y"))


build_date("10", "9", "2026")
build_date("31", "2", "2026")
build_date("ten", "9", "2026")

## 16.1 Student Date Activity

Modify the function so that it prompts for:

- day;
- month;
- year.

Requirements:

- use `int()` for each component;
- create a `date` object;
- catch invalid conversion or invalid calendar values;
- print the formatted date only when everything succeeds;
- print `"Date validation complete"` from `finally`.

In [ ]:
# Suggested solution — uncomment the calls to input() during class.

def interactive_date_validator():
    try:
        day = int(input("Day: "))
        month = int(input("Month: "))
        year = int(input("Year: "))
        chosen_date = date(year, month, day)
    except ValueError as exc:
        print("Invalid date:", exc)
    else:
        print("Accepted:", chosen_date.strftime("%d %B %Y"))
    finally:
        print("Date validation complete")

# interactive_date_validator()

# 17. Raising Exceptions Deliberately with `raise`

So far Python has raised exceptions automatically. We can also deliberately reject an invalid situation ourselves.

```python
if quantity < 0:
    raise ValueError("Quantity cannot be negative")
```

This is useful when a function has a clear contract and invalid input should not silently continue.

In [ ]:
def calculate_total(price, quantity):
    if price < 0:
        raise ValueError("Price cannot be negative.")
    if quantity < 0:
        raise ValueError("Quantity cannot be negative.")
    return price * quantity


for price, quantity in [(12.50, 3), (12.50, -1)]:
    try:
        print("Total:", calculate_total(price, quantity))
    except ValueError as exc:
        print("Order rejected:", exc)

# 18. Custom Exceptions

Built-in exceptions are often enough, but sometimes the program has a **domain-specific rule** that deserves a meaningful name.

A custom exception is usually a class that inherits from `Exception`:

```python
class DeliveryRangeError(Exception):
    pass
```

Now our code can explicitly communicate that the failure is related to delivery range rather than a generic numeric problem.

## 18.1 Custom Delivery Exception Flowchart

```text
             Receive delivery distance
                       │
                       ▼
              Convert value to float
                       │
             ┌─────────┴─────────┐
             │ conversion works? │
             └─────────┬─────────┘
                 No    │    Yes
                 │     │      │
                 ▼     │      ▼
             ValueError│   Is distance > 0
                       │   and <= service limit?
                       │      ┌────┴────┐
                       │     Yes        No
                       │      │          │
                       │      ▼          ▼
                       │   Accept     raise
                       │   delivery   DeliveryRangeError
                       │                  │
                       └──────────────────┘
                                  Handle clearly
```

In [ ]:
class DeliveryRangeError(Exception):
    """Raised when a requested delivery distance is outside our service area."""


MAX_DELIVERY_KM = 50


def calculate_delivery_fee(distance_text):
    try:
        distance = float(distance_text)
        if distance <= 0:
            raise DeliveryRangeError("Distance must be greater than 0 km.")
        if distance > MAX_DELIVERY_KM:
            raise DeliveryRangeError(
                f"Delivery is limited to {MAX_DELIVERY_KM} km."
            )
        fee = 5 + 0.80 * distance
    except ValueError:
        print("Distance must be numeric.")
    except DeliveryRangeError as exc:
        print("Delivery rejected:", exc)
    else:
        print(f"Delivery accepted. Fee = ${fee:.2f}")
    finally:
        print("Delivery check complete.")


calculate_delivery_fee("12")
calculate_delivery_fee("80")
calculate_delivery_fee("far")
calculate_delivery_fee("-3")

# 19. When Should We Create a Custom Exception?

Create a custom exception when it improves the meaning of your program.

| Situation | Good choice |
|---|---|
| Text cannot be converted to an integer | `ValueError` |
| Divide by zero | `ZeroDivisionError` |
| Missing dictionary key | `KeyError` |
| File is missing | `FileNotFoundError` |
| Delivery is outside your business's service area | Custom `DeliveryRangeError` |
| Student tries to enrol after a course-specific deadline | Potential custom `EnrollmentClosedError` |

A custom exception should describe a **meaningful application-level failure**, not merely rename every built-in exception.

# 20. Exception Handling Inside Functions

A useful design question is:

> **Should this function handle the exception, or should the caller handle it?**

Sometimes a low-level function should raise an exception and allow a higher-level part of the application to decide what message to show.

In [ ]:
def parse_positive_integer(text):
    value = int(text)  # may raise ValueError
    if value <= 0:
        raise ValueError("Value must be greater than zero.")
    return value


def application_layer(text):
    try:
        value = parse_positive_integer(text)
    except ValueError as exc:
        print("Could not accept input:", exc)
    else:
        print("Accepted value:", value)


application_layer("8")
application_layer("zero")
application_layer("-2")

# 21. Avoid These Common Exception-Handling Mistakes

### Mistake 1 — Catching everything and saying nothing

```python
try:
    ...
except Exception:
    pass
```

This hides bugs and makes debugging difficult.

### Mistake 2 — Putting too much code inside one `try`

If ten unrelated operations are inside a single `try`, it becomes harder to know which operation you intended to protect.

### Mistake 3 — Using exceptions instead of normal decisions

Do not raise an exception merely because a normal `if` condition is false.

### Mistake 4 — Printing an error but continuing with invalid state

Make sure your recovery path leaves the program in a valid state.

### Mistake 5 — Catching the wrong exception type

If `int()` raises `ValueError`, an `except TypeError:` block will not handle that failure.

In [ ]:
# Example: precise try block

def load_percentage(text):
    try:
        percentage = float(text)
    except ValueError:
        return None, "Percentage must be numeric."

    if not 0 <= percentage <= 100:
        return None, "Percentage must be between 0 and 100."

    return percentage, None


for sample in ["80", "eighty", "140"]:
    value, error = load_percentage(sample)
    print(sample, "->", error if error else value)

# 22. Guided Practice — Predict the Exception

Before running the next cell, predict the exception type for each operation.

1. `int("3.14")`
2. `[1, 2][8]`
3. `{"a": 1}["b"]`
4. `10 / 0`
5. `unknown_variable + 1`
6. `"age" + 20`

In [ ]:
examples = [
    ("int('3.14')", lambda: int("3.14")),
    ("[1, 2][8]", lambda: [1, 2][8]),
    ("{'a': 1}['b']", lambda: {"a": 1}["b"]),
    ("10 / 0", lambda: 10 / 0),
    ("unknown_variable + 1", lambda: eval("unknown_variable + 1")),
    ("'age' + 20", lambda: "age" + 20),
]

for expression, action in examples:
    try:
        action()
    except Exception as exc:
        print(f"{expression:<28} -> {type(exc).__name__}")

# 23. Guided Practice — Fix the Program

The following design is poor because it catches a very broad exception and gives the same message for all problems:

```python
try:
    age = int(age_text)
    score = 100 / age
except Exception:
    print("Error")
```

Rewrite it so that:

- non-integer age produces a clear input message;
- age `0` produces a clear division message;
- success prints the score;
- `finally` prints `"Processing complete"`.

In [ ]:
def fixed_program(age_text):
    try:
        age = int(age_text)
        score = 100 / age
    except ValueError:
        print("Age must be entered as a whole number.")
    except ZeroDivisionError:
        print("Age cannot be zero for this calculation.")
    else:
        print("Score =", score)
    finally:
        print("Processing complete")


fixed_program("25")
fixed_program("zero")
fixed_program("0")

# 24. Challenge — Robust Delivery Order Validator

Build a program that accepts:

- customer age;
- number of items;
- delivery distance;
- delivery day, month, and year.

Requirements:

1. Convert age and item count using `int()`.
2. Convert distance using `float()`.
3. Build the delivery date using `datetime.date`.
4. Handle conversion / date problems with `ValueError`.
5. Reject ages outside `0–130`.
6. Reject item counts less than `1`.
7. Raise `DeliveryRangeError` when distance is `<= 0` or greater than `50 km`.
8. Use `else` to print an order summary only after validation succeeds.
9. Use `finally` to print `"Order validation finished"`.

In [ ]:
def validate_delivery_order(age_text, item_text, distance_text,
                            day_text, month_text, year_text):
    try:
        age = int(age_text)
        item_count = int(item_text)
        distance = float(distance_text)
        delivery_date = date(int(year_text), int(month_text), int(day_text))

        if not 0 <= age <= 130:
            raise ValueError("Age must be between 0 and 130.")
        if item_count < 1:
            raise ValueError("Item count must be at least 1.")
        if distance <= 0 or distance > MAX_DELIVERY_KM:
            raise DeliveryRangeError(
                f"Distance must be greater than 0 and at most {MAX_DELIVERY_KM} km."
            )

    except DeliveryRangeError as exc:
        print("Delivery error:", exc)
    except ValueError as exc:
        print("Input error:", exc)
    else:
        print("Order accepted")
        print("Age          :", age)
        print("Items        :", item_count)
        print("Distance     :", distance, "km")
        print("Delivery date:", delivery_date.strftime("%d %B %Y"))
    finally:
        print("Order validation finished")


validate_delivery_order("28", "3", "12.5", "10", "9", "2026")
print()
validate_delivery_order("28", "3", "75", "10", "9", "2026")
print()
validate_delivery_order("twenty", "3", "12", "10", "9", "2026")

# 25. Exception Handling Decision Guide

Use this decision sequence when writing your own programs:

```text
1. What operation can fail?
        │
        ▼
2. What specific exception can it raise?
        │
        ▼
3. Can this part of the program recover meaningfully?
   ┌────┴────┐
  Yes        No
   │          │
   ▼          ▼
Catch the     Let the exception propagate
specific      to a layer that can handle it
exception
   │
   ▼
4. Is there success-only work?
   └─> put it in else
   │
   ▼
5. Is there cleanup that must always happen?
   └─> put it in finally
```

# 26. Mini Cheat Sheet

| Goal | Python structure |
|---|---|
| Protect an operation that may fail | `try:` |
| Handle a known failure | `except ValueError:` |
| Inspect the exception message | `except ValueError as exc:` |
| Handle different failures differently | multiple `except` blocks |
| Run code only after success | `else:` |
| Run cleanup regardless of success/failure | `finally:` |
| Trigger a built-in exception yourself | `raise ValueError("message")` |
| Define a domain-specific failure | `class MyError(Exception): ...` |

### Execution order

```text
SUCCESS:  try → else → finally
FAILURE:  try → except → finally
UNHANDLED: try → finally → exception propagates
```

# 27. 30-Question Self-Check Quiz

The quiz below covers the full Week 08 workbook. It uses `ipywidgets` if available and presents one question at a time.

In [ ]:
quiz_questions = [
    ("Which error means Python source code violates grammar rules?", ["ValueError", "SyntaxError", "KeyError", "RuntimeError"], 1),
    ("Which exception is raised by int('hello')?", ["TypeError", "NameError", "ValueError", "IndexError"], 2),
    ("Which exception is raised by 10 / 0?", ["ArithmeticError only", "ZeroDivisionError", "ValueError", "OverflowError"], 1),
    ("Which exception is commonly raised for an undefined variable name?", ["NameError", "KeyError", "AttributeError", "ImportError"], 0),
    ("Which exception is raised when a list index is outside its range?", ["KeyError", "IndexError", "RangeError", "LookupTypeError"], 1),
    ("Which exception is raised when a dictionary key is missing with d[key]?", ["NameError", "IndexError", "KeyError", "AttributeError"], 2),
    ("Which block contains code that may raise an exception?", ["else", "finally", "try", "raise"], 2),
    ("When does else run in a try statement?", ["Always", "Only if try succeeds", "Only if except runs", "Only if finally fails"], 1),
    ("When does finally normally run?", ["Only on success", "Only on failure", "On both success and failure", "Never after return"], 2),
    ("Why prefer ValueError over Exception when possible?", ["It is faster in every case", "It handles a more specific expected problem", "Exception cannot be caught", "ValueError catches syntax errors"], 1),
    ("What does 'as exc' do in except ValueError as exc?", ["Renames ValueError", "Stores the exception object in a variable", "Converts the exception to text", "Suppresses the error automatically"], 1),
    ("What keyword deliberately triggers an exception?", ["throw", "except", "raise", "error"], 2),
    ("A custom application exception normally inherits from...?", ["object only", "Exception", "str", "ValueError must always be used"], 1),
    ("What does TypeError usually signal?", ["A missing file", "An incompatible object type for an operation", "An invalid dictionary key only", "A syntax problem"], 1),
    ("What does ValueError usually signal?", ["Correct kind of input, unacceptable value", "Always an undefined name", "Only division by zero", "Only missing files"], 0),
    ("Which is the best handler for float('abc')?", ["except ValueError", "except KeyError", "except IndexError", "except ZeroDivisionError"], 0),
    ("Which block is best for success-only output after risky conversion?", ["try", "else", "finally", "raise"], 1),
    ("Which block is best for cleanup that must happen regardless?", ["except", "else", "finally", "if"], 2),
    ("What happens to the rest of a try block after an exception is raised inside it?", ["It always continues", "It is skipped while Python searches for a handler", "It becomes a finally block", "It restarts"], 1),
    ("If no except block matches, what normally happens?", ["The exception disappears", "The exception propagates", "else handles it", "Python converts it to ValueError"], 1),
    ("Which exception commonly represents a missing file?", ["FileNotFoundError", "PathValueError", "NameError", "KeyError"], 0),
    ("Which construct is usually cleaner than manual file close in finally?", ["while", "with open(...) as f", "global", "lambda"], 1),
    ("Is int('-4') a conversion error?", ["Yes, always", "No; conversion succeeds", "Only on Linux", "Only inside try"], 1),
    ("After int('-4') succeeds, checking whether -4 is a valid age is what?", ["Syntax parsing", "Validation", "Importing", "Indexing"], 1),
    ("What should a custom DeliveryRangeError represent?", ["Every numeric problem", "A domain-specific delivery range rule", "A syntax failure", "A missing Python module"], 1),
    ("Which sequence is correct when try succeeds with else and finally present?", ["try → finally → else", "try → else → finally", "else → try → finally", "try → except → else"], 1),
    ("Which sequence is correct when a matching exception occurs?", ["try → except → finally", "try → else → except", "except → try → finally", "try → finally → else"], 0),
    ("Why keep try blocks reasonably small?", ["Python has a 5-line limit", "To make it clear which operation is expected to fail", "else requires it", "finally cannot follow long blocks"], 1),
    ("Why is 'except: pass' usually poor practice?", ["It is invalid Python", "It can silently hide important problems", "It catches nothing", "It always raises SyntaxError"], 1),
    ("What are the two main Week 08 categories introduced at the start?", ["Compile and link errors", "Syntax errors and runtime errors", "HTML and Python errors", "Class and function errors"], 1),
]

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    state = {"index": 0, "score": 0, "answered": False}
    question_output = widgets.Output()
    feedback_output = widgets.Output()
    choices = widgets.RadioButtons(options=[])
    submit_button = widgets.Button(description="Submit answer")
    next_button = widgets.Button(description="Next question", disabled=True)

    def render_question():
        i = state["index"]
        q, options, answer = quiz_questions[i]
        choices.options = options
        choices.value = None
        state["answered"] = False
        submit_button.disabled = False
        next_button.disabled = True
        with question_output:
            clear_output(wait=True)
            print(f"Question {i + 1} of {len(quiz_questions)}")
            print(q)
        with feedback_output:
            clear_output(wait=True)

    def submit_answer(_):
        if state["answered"]:
            return
        if choices.value is None:
            with feedback_output:
                clear_output(wait=True)
                print("Choose an answer first.")
            return
        i = state["index"]
        q, options, answer = quiz_questions[i]
        selected_index = options.index(choices.value)
        state["answered"] = True
        submit_button.disabled = True
        next_button.disabled = False
        with feedback_output:
            clear_output(wait=True)
            if selected_index == answer:
                state["score"] += 1
                print("Correct.")
            else:
                print("Not quite. Correct answer:", options[answer])
            print(f"Current score: {state['score']} / {i + 1}")

    def next_question(_):
        if state["index"] + 1 >= len(quiz_questions):
            with question_output:
                clear_output(wait=True)
                print("Quiz complete!")
                print(f"Final score: {state['score']} / {len(quiz_questions)}")
                percentage = state["score"] / len(quiz_questions) * 100
                print(f"Percentage: {percentage:.1f}%")
            choices.options = []
            submit_button.disabled = True
            next_button.disabled = True
            with feedback_output:
                clear_output(wait=True)
            return
        state["index"] += 1
        render_question()

    submit_button.on_click(submit_answer)
    next_button.on_click(next_question)
    display(widgets.VBox([
        question_output,
        choices,
        widgets.HBox([submit_button, next_button]),
        feedback_output,
    ]))
    render_question()

except Exception as exc:
    print("Interactive quiz unavailable:", exc)
    print("You can still review quiz_questions directly.")

# 28. Final Summary

This week introduced a complete model for dealing with failures in Python:

1. **Syntax errors** mean the source code itself is invalid.
2. **Runtime errors** happen while valid code is executing.
3. Python represents runtime failures using **exception objects**.
4. `try` identifies code that may fail.
5. `except` handles a matching failure.
6. `else` runs only when the `try` block succeeds.
7. `finally` runs regardless of success or failure and is useful for cleanup.
8. Specific exception types communicate more information than a broad catch.
9. `raise` lets our own code enforce a rule.
10. Custom exception classes communicate domain-specific failures.
11. Conversion and validation are related but different: converting `"-4"` to `-4` succeeds, but the application can still reject the resulting value.
12. Good exception handling should make a program **clearer, safer, and easier to debug** — not hide bugs.

---

## Before next week

Make sure you can write from memory:

```python
try:
    ...
except ValueError as exc:
    ...
else:
    ...
finally:
    ...
```

Then practise deciding **which exception type** belongs in the `except` block and **which validation belongs in normal program logic**.